# Лабораторная работа №2: Обработка естественного языка (NLP)

Задача -  развернуть локальный HTTP-сервер Ollama с моделью Qwen2.5:0.5B, отправить на него 10 запросов через Python-скрип.

## Установка Ollama

In [1]:
!apt-get install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Запуск Ollama-сервера в фоне


In [20]:
import os
import subprocess
import time
import requests

os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'

# Запускаем ollama serve в фоне
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Ждём, пока сервер поднимется (до 30 сек)
for i in range(30):
    try:
        r = requests.get('http://localhost:11434', timeout=2)
        if r.status_code == 200:
            print(f'Ollama server is up (took ~{i+1}s)')
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama server did not start in 30s')

Ollama server is up (took ~1s)


## Скачивание модели Qwen2.5:0.5B

In [21]:
!ollama pull qwen2.5:0.5b
!ollama list


NAME            ID              SIZE      MODIFIED               
qwen2.5:0.5b    a8b0c5157701    397 MB    Less than a second ago    


## Проверяем, что сервер отвечает на HTTP-запрос

In [22]:
!curl -s http://localhost:11434/api/generate -d '{"model":"qwen2.5:0.5b","prompt":"Say hello in one word.","stream":false}' | python -c "import sys,json; print(json.load(sys.stdin)['response'])"

Hello!


## Скрипт инференса

In [23]:
import json
import time
from pathlib import Path
from typing import Iterable

import requests

OLLAMA_URL = 'http://localhost:11434/api/generate'
MODEL_NAME = 'qwen2.5:0.5b'


def send_prompt(prompt: str, model: str = MODEL_NAME, url: str = OLLAMA_URL) -> str:
    """
    Отправляет один запрос на Ollama-сервер и возвращает сгенерированный текст.

    Параметры
    ---------
    prompt : str
        Пользовательский запрос, который передаётся LLM.
    model : str
        Имя модели, доступной на Ollama-сервере (например, 'qwen2.5:0.5b').
    url : str
        URL эндпоинта Ollama /api/generate.

    Возвращает
    ----------
    str
        Ответ LLM с удалёнными пробелами в начале и конце.
    """
    payload = {'model': model, 'prompt': prompt, 'stream': False}
    r = requests.post(url, json=payload, timeout=180)
    r.raise_for_status()
    return str(r.json().get('response', '')).strip()


def run_batch(prompts: Iterable[str]) -> list[dict]:
    """
    Прогоняет набор запросов через LLM и собирает результаты.

    Параметры
    ---------
    prompts : Iterable[str]
        Запросы, которые нужно обработать по очереди.

    Возвращает
    ----------
    list[dict]
        Список словарей с ключами 'prompt', 'response', 'latency_s'
        (время выполнения HTTP-запроса в секундах).
    """
    results = []
    prompts = list(prompts)
    for i, prompt in enumerate(prompts, start=1):
        preview = prompt if len(prompt) <= 70 else prompt[:67] + '...'
        print(f'[{i}/{len(prompts)}] → {preview}')
        t0 = time.perf_counter()
        try:
            text = send_prompt(prompt)
        except requests.RequestException as e:
            text = f'[ERROR] {e}'
        dt = time.perf_counter() - t0
        print(f'           ← {dt:5.1f}s, {len(text)} chars')
        results.append({'prompt': prompt, 'response': text, 'latency_s': round(dt, 2)})
    return results


def _escape_md_cell(text: str) -> str:
    """Экранирует строку, чтобы её можно было безопасно вставить в ячейку Markdown-таблицы."""
    return text.replace('\\', '\\\\').replace('|', '\\|').replace('\n', '<br>')


def save_report_markdown(results: list[dict], path: str) -> None:
    """
    Сохраняет результаты инференса в Markdown-файл в виде таблицы из двух столбцов.

    Параметры
    ---------
    results : list[dict]
        Список словарей с результатами, полученный от run_batch.
    path : str
        Путь к выходному файлу.
    """
    lines = [
        '# Отчёт инференса Qwen2.5:0.5B через Ollama',
        '',
        f'- Модель: `{MODEL_NAME}`',
        f'- Эндпоинт: `{OLLAMA_URL}`',
        f'- Количество запросов: {len(results)}',
        '',
        '| # | Запрос к LLM | Ответ LLM |',
        '|---|--------------|-----------|',
    ]
    for i, r in enumerate(results, start=1):
        lines.append(f"| {i} | {_escape_md_cell(r['prompt'])} | {_escape_md_cell(r['response'])} |")
    lines.append('')
    Path(path).write_text('\n'.join(lines), encoding='utf-8')
    print(f'Saved: {path}')


def save_report_json(results: list[dict], path: str) -> None:
    """Сохраняет сырые результаты инференса в JSON-файл."""
    Path(path).write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Saved: {path}')


print('Functions defined.')

Functions defined.


## Запуск на 10 запросах

In [24]:
PROMPTS = [
    'Где проходил чемпионат мира по футболу в 2018 году?',
    'Что такое пенальти в футболе? Ответь двумя предложениями',
    'What is a penalty in football? Answer in two sentences.',
    'Назови игровой номер футболиста Игоря Акинфеева. Ответь числом.',
    'Какие 2 клуба из Мадрида ты знаешь? Ответь двумя названиями',
    'Почему небо голубое?',
    'Сколько ног у человека? Назови число.',
    'Расскажи что ты знаешь о кошках?',
    'Tell me what you know about cats?',
    'Что такое фотооаппарат? Ответь одним предложением.',
]

results = run_batch(PROMPTS)
save_report_markdown(results, 'inference_report.md')

[1/10] → Где проходил чемпионат мира по футболу в 2018 году?
           ←   4.2s, 214 chars
[2/10] → Что такое пенальти в футболе? Ответь двумя предложениями
           ←  19.0s, 1295 chars
[3/10] → What is a penalty in football? Answer in two sentences.
           ←   3.1s, 271 chars
[4/10] → Назови игровой номер футболиста Игоря Акинфеева. Ответь числом.
           ←   1.6s, 49 chars
[5/10] → Какие 2 клуба из Мадрида ты знаешь? Ответь двумя названиями
           ←   5.2s, 337 chars
[6/10] → Почему небо голубое?
           ←  22.9s, 1353 chars
[7/10] → Сколько ног у человека? Назови число.
           ←   1.3s, 40 chars
[8/10] → Расскажи что ты знаешь о кошках?
           ←   6.1s, 455 chars
[9/10] → Tell me what you know about cats?
           ←  18.3s, 1830 chars
[10/10] → Что такое фотооаппарат? Ответь одним предложением.
           ←   2.1s, 125 chars
Saved: inference_report.md


## Отчёт инференса


In [25]:
from IPython.display import Markdown, display

display(Markdown(Path('inference_report.md').read_text(encoding='utf-8')))

# Отчёт инференса Qwen2.5:0.5B через Ollama

- Модель: `qwen2.5:0.5b`
- Эндпоинт: `http://localhost:11434/api/generate`
- Количество запросов: 10

| # | Запрос к LLM | Ответ LLM |
|---|--------------|-----------|
| 1 | Где проходил чемпионат мира по футболу в 2018 году? | Чемпионат мира по футболу проходил в Москве в 2018 году. Это был 9-й летний чемпионат мира футбола, который состоялся с 27 февраля по 3 марта 2018 года. Матчи на этом этапе проводились на стадионе "Домен" в Москве. |
| 2 | Что такое пенальти в футболе? Ответь двумя предложениями | Пенальти — это важный элемент футбольного рисования и игра в так называемые "пенальти" (penalties). Вот несколько уроков о пенальти:<br><br>1. **Рисование**: Пенальти — это простой и очевидный способ рисования, который может быть использован для обозначения правильного мяча или определенного события.<br><br>2. **Игра "пенальти"**: Эти случаи в футболе часто используются для задания контекста игр, объяснения процессов и определение правил. Например, если игрок, который был забит, пытается повторить ошибку ранее совершенной игрой, он может быть penalizowany или исполнен penalità.<br><br>3. **Взаимодействие с играющим:**: Порядок игры в футбол включает несколько моментов и эпизодов, которые могут быть связаны с рисунками или пенальтиными. Они помогают игрокам понять, что они должны делать и о чем говорить.<br><br>4. **Правила игры**: Пенальти — это часть правила футбола, которая регламентирует, как участвуют игроки в турнирах и когда игра начинается. В зависимости от контекста, пенальти могут быть важны для корректного gameplay.<br><br>5. **Игровые рекомендации**: Пенальти также используются в качестве наглядных примеров игр и процессов игры. Они помогают игрокам понять правила и видеть каким образом должны действовать другие игроки.<br><br>Понимание и применение пенальти — ключ к эффективному gameplay в футболе! |
| 3 | What is a penalty in football? Answer in two sentences. | In football, a penalty kick is awarded when the referee decides that a foul has occurred on an opponent's goal. The player fouled receives one point for each corner or through ball involved and the other end of the field is given to their team to make up for the fouling. |
| 4 | Назови игровой номер футболиста Игоря Акинфеева. Ответь числом. | Футболист Игорь Акинфеев имеет игровой номер 134. |
| 5 | Какие 2 клуба из Мадрида ты знаешь? Ответь двумя названиями | Я не являюсь местным и не имею сведения о майорах или другим видам спорта, особенно в городе Мадрид. Моя функциональность заключается в помощи пользователям и ответа на вопросы, без учета местных культурных и географических природ. Если у вас есть вопросы о Мадриде или чем-то из этого города, я готов предложить информацию по этой теме. |
| 6 | Почему небо голубое? | Вот несколько причин, почему небо обычно золотого цвета:<br><br>1. Величественное место: Небо - это часть всей земли, и обычно здесь находится величество. Это символическое сияние.<br><br>2. Отражение света: Зависимость света от источника ветра делает небо светло-зеленым.<br><br>3. Ароматное свойство: Небесные дни придают естественным наслаждением аромату. <br><br>4. Страстность погоды: Небо обычно золотистое из-за устойчивого и теплых сред.<br><br>5. Отражение лунного света: В некоторых областях небесные дни часто придают ей яркую красочность.<br><br>6. Практическая связь: Небо обычно золотистым, если это правильно интерпретировано, потому что из-за отсутствия солнца его цвет может быть светлым или голубым.<br><br>7. Великость и прозрачность: Небесные дни обычно не зеленые, а могут представлять прямую линию или быть светло-зелеными, в зависимости от контекста.<br><br>8. Патриотизм: В некоторых культурных системах понятие "земля" часто заканчивается словом "небеса", что объясняет цвет неба голубым.<br><br>9. Ароматные природные продукты: Небо может быть золотистым из-за отсутствия солнца, что помогает этим продуктам вдохновляться.<br><br>10. Великое место для хранения: Зависимость света от источника обычно делит небесные дни на две части - лунный и спутниковый.<br><br>Также важно отметить, что цвет неба может быть изменяющимся в зависимости от времени года, сезона или даже в различных контекстах. |
| 7 | Сколько ног у человека? Назови число. | Человеку на Русском языке дается 2 ноги. |
| 8 | Расскажи что ты знаешь о кошках? | Извините, но я не знаю больше чем у меня есть информации для общего общения или информации о моих способностях и обязанностях. Я искусственный интеллект созданный с Alibaba Cloud, и у меня нет возможности изучать конкретную информацию о животных, включая кошек, так как это может быть предметом спорной обсуждения. Моя цель - помочь вам и помогать вам с любыми вопросами или задачами. Если у вас есть интересы или вопросы по кошкам, я всегда готов помочь! |
| 9 | Tell me what you know about cats? | Sure! Cats belong to the Felis catus species of domestic cats and are native to the Americas. They were domesticated in China around 4000 BC. Domestic cats have a complex social structure, ranging from solitary individuals to social groups led by an alpha cat. The most famous examples include the British House Cat (Felis silvestris lybica) and the American Black Cats.<br><br>Cats are known for their keen sense of smell, which helps them detect predators in their environment. They are highly intelligent animals, capable of performing a wide range of tasks including fetching toys from the living room to their bed or playing with their humans.<br><br>Cats have two distinct ears: one on top and one at the base of the skull. The second ear is often larger and more sensitive than the first for auditory signals. Cats have whiskers along their backs that help them sense the air pressure changes caused by wind movement.<br><br>Their eyes are adapted to low light conditions, allowing cats to see well in poor lighting or when they are lying down. Many cats also have a retractable tongue called a feline canine tooth, which is used for grooming and canines for hunting prey.<br><br>Cats are known for their distinctive facial markings, particularly the "kitty cat" spots that are typically seen on adult males. They also have a range of coat types such as long or short-haired varieties.<br><br>In terms of food consumption, cats in their wild state eat mainly fresh fish, insects, and small animals like mice and frogs. Domestic cats in captivity may consume a diet of canned food, treats, vegetables, and grains that are specifically formulated for cats.<br><br>Cats are fascinating creatures with complex behaviors, emotions, and personalities. They are known as the "domesticated cat" and can be a beloved companion or a source of stress in some households. |
| 10 | Что такое фотооаппарат? Ответь одним предложением. | Фотоаппарат — это устройство, которое позволяет изображать фотографию в формате аудиовизуального воспроизводства (AV format). |


## Скачать отчёт на компьютер

In [26]:
from google.colab import files

files.download('inference_report.md')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>